# World of Shadow Work — high-quality vessel splats (`vessel.wswv`)

Generates **more, higher-quality** image-to-3D Gaussian splats for the runtime's morphing "vessel", using **TRELLIS** (Microsoft's image-to-3D structured-latent model — the best 3DGS quality). One splat keyframe per input image; the runtime melts between them live.

**Runtime: A100 80GB.** TRELLIS only needs ~16–24 GB, so 80 GB lets us run it at full sampling quality and batch many images comfortably.

**Pipeline:** corpus images → TRELLIS → one `.ply` (3DGS) per image → `stage_d_vessel.py` packs them (Morton-ordered, fixed-G, one global AABB) into `assets/vessel.wswv` (the compact `WSWV` binary the allolib runtime loads — **no runtime ML**).

**HARD RULE (offline factory):** this notebook **downloads** the finished `vessel.wswv` to your Mac. It must **never git-push** assets from Colab. Drop the downloaded file into `reagency/assets/vessel.wswv` on the Mac; the runtime auto-loads it.

> Fallback if TRELLIS install is fussy: LGM (feed-forward, trivial on an A100) — see the last section + `VESSEL.md`.

## 0 · Confirm the A100 80GB
Runtime → Change runtime type → **A100 GPU**. Expect `A100-SXM4-80GB`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.get_device_name(0))

## 1 · Get the repo + a subset of corpus images (the splat keyframes)
Only the chosen keyframe images are fetched (straight from `ATTRIBUTION.csv` `image_url`s) — not the whole 14k corpus. Tune `N_IMAGES` / `SELECT`.

`N_IMAGES` = number of splat keyframes (more = richer morph, bigger file). `SELECT`: `spread` (evenly across the corpus), `first` (first N), or `dreams` (the `aic_*` dream-source artworks).

In [ ]:
import os, csv, urllib.request, time, glob, shutil

REPO   = 'https://github.com/9LiveZZZ-Git/MAT201B_Projects.git'
BRANCH = 'v2'
if not os.path.exists('/content/MAT201B_Projects'):
    !git clone -b {BRANCH} {REPO} /content/MAT201B_Projects
FACTORY = '/content/MAT201B_Projects/reagency/factory'
CORPUS  = f'{FACTORY}/corpus'

# ---- config -------------------------------------------------------------
N_IMAGES = 96          # number of splat keyframes to generate
SELECT   = 'spread'    # 'spread' | 'first' | 'dreams'
# -------------------------------------------------------------------------

rows   = list(csv.DictReader(open(f'{CORPUS}/ATTRIBUTION.csv', newline='', encoding='utf-8')))
files  = [r['file'] for r in rows]
url_of = {r['file']: r.get('image_url', '').strip() for r in rows}
if   SELECT == 'first':  picks = files[:N_IMAGES]
elif SELECT == 'dreams': picks = [f for f in files if os.path.basename(f).startswith('aic_')][:N_IMAGES]
else:                    picks = files[::max(1, len(files)//N_IMAGES)][:N_IMAGES]

INPUT_DIR = f'{CORPUS}/_vessel_inputs'
os.makedirs(INPUT_DIR, exist_ok=True)
UA = {'User-Agent': 'WorldOfShadowWork/0.1 (MAT201B academic art project; lpfreiburg@ucsb.edu)'}
got = 0
for rel in picks:
    u = url_of.get(rel, '')
    if not u:
        continue
    dst = f'{INPUT_DIR}/{os.path.basename(rel)}'
    if not (os.path.exists(dst) and os.path.getsize(dst) > 0):
        try:
            req = urllib.request.Request(u, headers=UA)
            open(dst, 'wb').write(urllib.request.urlopen(req, timeout=30).read())
            time.sleep(0.05)
        except Exception as e:
            print('ERR', rel, e); continue
    got += 1
print(f'{got} keyframe images ready in {INPUT_DIR}')

## 2 · Install TRELLIS (highest quality; comfortable on the 80GB A100)
Repo: https://github.com/microsoft/TRELLIS . The official `setup.sh` installs the CUDA-extension stack into the current env. ~5–10 min the first time.

If a flag fails, drop it (e.g. `--flash-attn` → rely on `--xformers`; the run cell sets `ATTN_BACKEND` accordingly).

In [ ]:
import os
if not os.path.exists('/content/TRELLIS'):
    !git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS
%cd /content/TRELLIS
# install into the existing Colab env (no new conda env); pick the backends that build cleanly on the A100
!. ./setup.sh --basic --xformers --spconv --mipgaussian --nvdiffrast --diffoctreerast --kaolin
# rembg backend for TRELLIS's built-in background removal
!pip -q install rembg onnxruntime imageio imageio-ffmpeg plyfile

## 3 · Run TRELLIS → one 3DGS `.ply` per image
Each input image becomes a 3D Gaussian splat keyframe. `STEPS` higher = cleaner (the 80 GB lets us push it). Saves to `factory/work/vessels/*.ply` (re-runnable — existing `.ply`s are skipped).

In [ ]:
import os, glob
os.environ['ATTN_BACKEND'] = 'xformers'   # set to 'flash-attn' if you installed it
os.environ['SPCONV_ALGO']  = 'native'     # avoids first-run autotune stalls
%cd /content/TRELLIS
from PIL import Image
from trellis.pipelines import TrellisImageTo3DPipeline

pipe = TrellisImageTo3DPipeline.from_pretrained('microsoft/TRELLIS-image-large')
pipe.cuda()

VESSELS = f'{FACTORY}/work/vessels'
os.makedirs(VESSELS, exist_ok=True)
STEPS = 25   # sparse-structure + SLAT sampling steps (12 = fast, 25–50 = high quality)

imgs = sorted(glob.glob(f'{INPUT_DIR}/*.jpg') + glob.glob(f'{INPUT_DIR}/*.png'))
print(f'{len(imgs)} images -> splats')
for i, ip in enumerate(imgs):
    name = os.path.splitext(os.path.basename(ip))[0]
    out  = f'{VESSELS}/{name}.ply'
    if os.path.exists(out) and os.path.getsize(out) > 0:
        continue
    try:
        image = Image.open(ip).convert('RGB')
        outputs = pipe.run(
            image, seed=1,
            sparse_structure_sampler_params={'steps': STEPS, 'cfg_strength': 7.5},
            slat_sampler_params={'steps': STEPS, 'cfg_strength': 3.0},
        )
        outputs['gaussian'][0].save_ply(out)
        print(f'[{i+1}/{len(imgs)}] {name}.ply  {os.path.getsize(out)/1e6:.1f} MB')
    except Exception as e:
        print(f'[{i+1}/{len(imgs)}] SKIP {name}: {e}')
print('done. .ply keyframes in', VESSELS, '->', len(glob.glob(VESSELS+"/*.ply")))

## 4 · Pack → `assets/vessel.wswv` (the existing Stage-D packer)
`stage_d_vessel.py` prunes each keyframe to a fixed `G`, normalizes all to one global AABB, and Morton-orders them so the runtime's index-lerp is a real morph (not swimming). It auto-reduces `G` to keep the file under `--max-mb`.

**On `G` vs the runtime densifier:** the runtime (`viz/VesselSplats.cpp`) currently fans each baked gaussian into `DENS=160` satellites (so a *sparse* baked cloud reads dense). TRELLIS clouds are already dense, so for max quality use a **higher `G`** here **and lower `DENS`** on the Mac (e.g. `DENS=4`) — otherwise keep `G` modest (≈4000) to match `DENS=160`. Pick one and stay consistent.

In [ ]:
%cd {FACTORY}
G = 4000   # ~4000 x DENS160 ≈ 640k live points (dome-ish). For real-3DGS quality raise to ~30000 and set DENS=4 on the Mac.
!python3 stage_d_vessel.py --plys work/vessels --G {G} --max-mb 60
OUT = '/content/MAT201B_Projects/reagency/assets/vessel.wswv'
import os; print('vessel.wswv:', round(os.path.getsize(OUT)/1e6, 2), 'MB')

## 5 · Download to the Mac  (NEVER git-push from Colab)
Save `vessel.wswv`, then on the Mac drop it into `reagency/assets/vessel.wswv` — the runtime tries `assets/vessel.wswv` first, so it auto-loads on next launch.

In [ ]:
from google.colab import files
files.download(OUT)
# --- or stage it on Drive instead of a browser download: ---
# from google.colab import drive; drive.mount('/content/drive')
# import shutil, os; os.makedirs('/content/drive/MyDrive/wosw', exist_ok=True)
# shutil.copy(OUT, '/content/drive/MyDrive/wosw/vessel.wswv'); print('staged on Drive')

## Fallback — LGM (if TRELLIS install fails)
LGM (3DTopia/LGM, ECCV 2024) is feed-forward, a few seconds/image on an A100, and emits 3DGS `.ply` that `stage_d_vessel.py` reads the same way. Lower fidelity than TRELLIS but rock-solid. Run instead of cells 2–3, then continue at cell 4.
```bash
%cd /content
!pip -q install -U xformers plyfile
!git clone --recursive https://github.com/ashawkey/diff-gaussian-rasterization && pip -q install ./diff-gaussian-rasterization
!pip -q install git+https://github.com/NVlabs/nvdiffrast
!git clone https://github.com/3DTopia/LGM && cd LGM && pip -q install -r requirements.txt
# download pretrained/model_fp16.safetensors per the LGM README, then:
!cd LGM && python infer.py big --resume pretrained/model_fp16.safetensors \
    --workspace /content/MAT201B_Projects/reagency/factory/work/vessels \
    --test_path {INPUT_DIR}     # LGM rembg's the background itself
```
(See `factory/VESSEL.md` for the full LGM runbook.)